# 2026 Spring “EE545 Introductory Quantum Mechanics ”

## Time-Inependent Schrödinger Equation (TISE) Simulation

Update history
- 2021.01.01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code. (v1)
- 2024.06.26 : Minsu Jeong,  KAIST Electrical Engineering, minor revision and clean up
- 2025.03.27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization and the flexibility of the shape of the potential. (v2)
- 2025.09.01 : Minsu Jeong,  KAIST Electrical Engineering, revision and clean up

Reference :
- Griffith, Introduction to Quantum Mechanics, 3ed (2018)
- Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p31
- John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)

---

## Basic Formulation

- The TISE describes the stationary states of a quantum system and is given by:  
  $$
  \hat{H}\psi(x) = E\psi(x),
  $$  
  where:
  - $\hat{H}$ is the Hamiltonian operator.
  - $\psi(x)$ is the eigenfunction (wavefunction).
  - $E$ is the corresponding energy eigenvalue.

- For a single particle in one dimension, the Hamiltonian typically includes the kinetic and potential energy terms:  
  $$
  \hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + V(x).
  $$  
  Here, the kinetic term is represented by the second derivative with respect to $x$, and $V(x)$ is the potential energy function.



## Discretization on a Real Space Grid

- **Grid Representation:**  
  The continuous space is discretized into a grid with spacing $\Delta x$. The wavefunction $\psi(x)$ is then represented by its values on these grid points.

- **Finite Difference Approximation:**  
  The second derivative $\frac{d^2}{dx^2}$ in the kinetic term is approximated using finite difference methods. For example, a central difference scheme may be used:
  $$
  \frac{d^2\psi}{dx^2} \approx \frac{\psi(x+\Delta x) - 2\psi(x) + \psi(x-\Delta x)}{(\Delta x)^2}.
  $$

- **Hamiltonian Matrix Construction:**  
  Using the finite difference approximation, the Hamiltonian becomes a matrix where:
  - The diagonal elements typically contain $V(x)$ plus the central finite difference coefficient.
  - The off-diagonal elements represent the coupling between neighboring grid points due to the kinetic term.



## Eigenvalue Problem

- Once the Hamiltonian matrix is constructed, the TISE reduces to an eigenvalue problem:
  $$
  H \vec{\psi} = E \vec{\psi},
  $$  
  where $\vec{\psi}$ is the discretized wavefunction vector.

- **Numerical Solving:**  
  Standard numerical methods (e.g., using linear algebra libraries) can be used to solve for the eigenvalues $E$ and eigenvectors $\vec{\psi}$.  
  These eigenvalues represent the allowed energy levels of the system, while the eigenvectors provide the spatial distribution of the quantum states.



## Boundary Conditions

- **Dirichlet Boundary Conditions:**  
  For finite systems, the wavefunction is set to zero at the boundaries:
  $$
  \psi(x=0) = \psi(x=L) = 0.
  $$  
  This condition mimics an infinite potential outside the domain.



---
# (1) Calculation

In [ ]:
# install required libraries (run once)
import sys
print("Python executable:", sys.executable)

!{sys.executable} -m pip install -q --upgrade numpy scipy matplotlib ipython ipywidgets widgetsnbextension jupyterlab_widgets plotly nbformat tqdm

import ipywidgets as widgets
print("ipywidgets:", widgets.__version__)

import nbformat
print("nbformat:", nbformat.__version__)

Python executable: /home/class/anaconda3/envs/env1/bin/python
ipywidgets: 8.1.8
nbformat: 5.10.4


In [ ]:
"""
Update history
2021. 01. 01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code.
2024. 06. 26 : Minsu Jeong,  KAIST Electrical Engineering, minor revision and clean up
2025. 03. 27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization and the flexibility of the shape of the potential.
2025. 09. 01 : Minsu Jeong,  KAIST Electrical Engineering, revision and clean up

Reference :
- Griffith, Introduction to Quantum Mechanics, 3ed (2018)
- Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p31
- John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)
"""


import numpy as np
import numpy.linalg as lin
import math
import ipywidgets as widgets
from ipywidgets import interact_manual
from IPython.display import HTML, display
import plotly.graph_objects as go
import plotly.express as px
from scipy.constants import physical_constants

# Set numpy print options for debugging (optional)
np.set_printoptions(threshold=784, linewidth=np.inf)

# Constant and unit conversion
hbar = physical_constants["reduced Planck constant"][0]

bohr_radius     = physical_constants['Bohr radius'][0]  # in miter
angstrom_radius = 1e-10                                 # in miter
ang2bohr        = 1e-10 / bohr_radius  # ~1.88973

hartree_energy  = physical_constants['Hartree energy'][0]   # in Jule
ev_energy       = physical_constants['electron volt'][0]    # in Jule
har2ev          = hartree_energy / ev_energy  # ~27.21140795


def laplacianCoeff(laplacianDim: int):
    """
    Compute finite-difference weights for approximating the second derivative (Laplacian) on a uniform grid
    using the method of undetermined coefficients (moment conditions).

    Constructs and solves the linear Vandermonde-like system A b = c, where
      A_{i,j} = (j - m)^i / i!, for i = 0..2m, j = 0..2m,
      and c corresponds to the second-derivative moment (c[2] = 2!).
    Only nonnegative offsets are returned; symmetry implies weights for negative offsets are identical.
    Complexity: O(m^3) for matrix inversion; suitable for moderate m.

    ref - Fornberg, B. Mathematics of Computation, 51(184), 699-706. (1988)

    input:
        int         laplacianDim    Accuracy (dimension) of the Laplacian operator
    output:
        np.array    lcoeff          Coefficients
    """
    a = np.zeros((2*laplacianDim+1, 2*laplacianDim+1))
    c = np.zeros(2*laplacianDim+1)
    c[2] = math.factorial(2)
    for i in range(2*laplacianDim+1):
        for j in range(2*laplacianDim+1):
            a[i, j] = (j - laplacianDim)**i
    inva = lin.inv(a)
    b = np.matmul(inva, c)
    lcoeff = b[laplacianDim:2*laplacianDim+1]
    return lcoeff

def laplacian(ngridx, laplacianDim, dx):
    """
    Construct the discrete Laplacian operator matrix on a periodic one-dimensional grid.

    The operator uses a central finite-difference stencil of width 2m+1,
    where m = laplacian_order

    ref - Fornberg, B. Mathematics of Computation, 51(184), 699–706. (1988)

    input:
        int         ngridx          Number of spatial grid points
        int         laplacianDim    Accuracy (dimension) of the Laplacian operator
        float       dx              Grid spacing
    output:
        np.array    loprt           Laplacian operator matrix
    """
    lcoeff = laplacianCoeff(laplacianDim)
    loprt = np.zeros((ngridx, ngridx))
    for i in range(ngridx):
        for j in range(-laplacianDim, laplacianDim + 1):
            k = i + j
            if k >= ngridx:
                k -= ngridx
            elif k < 0:
                k += ngridx
            loprt[i][k] = lcoeff[abs(j)] / (dx**2)
    return loprt

def potential(ngridx, pot_shape=0, pot_height_har=25, curvature=0.5, width=0, slope=0.5, distance=0, dx=0):
    """
    Set the shape of potential V and return the potential on a grid.

    input:
        int         ngridx              Number of spatial grid points
        int         pot_shape           Shape of potential:
                                         0 = No potential (Box)
                                         1 = Step potential
                                         2 = Single wall
                                         3 = Double wall
                                         4 = Finite well (packet starts at middle)
                                         5 = Harmonic well
        int         pot_height_har       Height of the potential barrier (Hartree)
        int         barrier_thicknss_bohr    Thickness of the potential barrier (Bohr)
    output:
        np.array    pot_grid            Potential on the grid
    """

    # pot_height_har = pot_height_eV / eff_har2ev

    # Initialize potential grid
    pot_grid = np.zeros(ngridx)
    pot_grid[0] = 1e9
    pot_grid[ngridx-1] = 1e9

    width    = int(width/dx)
    center   = int(ngridx * 0.5)
    distance = int(distance/dx)
    

    if pot_shape == 0:      # Finite square well
        '''
                     width
                     <---->
        -----------─┐     ┌------------ height
                    │     │     
                    │     │      
                    └-----┘             0
        ---------------┼--------------->
                    center
        '''
        pot_grid[1:ngridx] = pot_height_har
        pot_grid[center - width // 2 : center + width // 2] = 0

    elif pot_shape == 1:    # infinite square well
        '''
                    │     │
                    │width│             ↑ inf
                    │<--->│
                    │     │
                    │     │     
                    │     │     
                    └-----┘             0
        ---------------┼--------------->
                    center
        '''
        pot_grid[1:ngridx] = 1e9
        pot_grid[center - width // 2 : center + width // 2] = 0    

    elif pot_shape == 2:    # Symmetric double square well
        '''
              width distance width
              <----><------><---->
        -----─┐     ┌-------┐     ┌----- height
              │     │       │     │
              │     │       │     │
              └-----┘       └-----┘      0
        ----------------┼---------------->
                     center
               
        '''
        pot_grid[1:ngridx] = pot_height_har
        pot_grid[center - width - distance // 2: center - distance // 2] = 0
        pot_grid[center + distance // 2: center + width + distance // 2] = 0

    elif pot_shape == 3:    # Harmonic
        '''
        y=a(x-b)^2

        curvature = y'' = 2a
        '''
        for i in range(1, ngridx):
            pot_grid[i] = (i - (ngridx-1)//2)**2 / (((ngridx-1)//2)**2) # normalize
        pot_grid[1:ngridx] = pot_grid[1:ngridx] * pot_height_har * curvature

    elif pot_shape == 4:    # Triangular
        '''
            │   /
            │  /
            │ /
            │/
        ----┼---------------->
           vertex
        '''
        center = int(ngridx * 0.5)
        tri_width = max(2, width)
        vertex = center - tri_width // 2
        left = max(1, vertex)
        right = min(ngridx - 1, vertex + tri_width)
        pot_grid[:] = 1e6
        for i in range(0, max(0, right - left)):
            pot_grid[left + i] = i * dx * slope

    elif pot_shape == 5:    # Asymmetric potential (1)
        '''
                 │width
                 │<---->
                 │     ┌---------- height
                 │     │
                 │     │
                 └-----┘           0
        ------------┼--------------->
                 center
        '''

        pot_grid[1 : width] = 0  
        pot_grid[width: ngridx - 1] = pot_height_har  
    
    elif pot_shape == 6:    # Asymmetric potential (2)
        '''
           │
           │width distance width
           │<----><------><---->
           │     ┌-------┐     ┌----- height
           │     │       │     │
           │     │       │     │
           └-----┘       └-----┘      0
        -------------┼---------------->
                 center
        '''

        pot_grid[1 : width] = 0  
        pot_grid[width : width + distance] = pot_height_har 
        pot_grid[width + distance : 2*width + distance] = 0
        pot_grid[2*width + distance: ngridx - 1] = pot_height_har 

    elif pot_shape == 7:    # Custum potential
        pot_grid[1 : ngridx - 1] = 0  

    return pot_grid

def hamiltonian(ngridx, laplacianDim, pot_height_har=25, pot_shape=0, 
                curvature=0.5, width=0, slope=0.6, distance=0, dx=0):
    """
    Define the Hamiltonian using the Laplacian and Potential operators.

    H = - (1 / 2) ∇^2 + V

    """
    # Setup potential operator
    pot_op = np.zeros((ngridx, ngridx))
    pot_1d = potential(ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx)
    np.fill_diagonal(pot_op, val=pot_1d)

    # Setup Laplacian operator
    laplacian_op = laplacian(ngridx, laplacianDim, dx)

    # Setup Hamiltonian operator with effective mass
    ham = - 1.0 * laplacian_op / 2. + pot_op
    return ham

def solve_tise(pot_shape=0, curvature=0.5, width=0, slope=0.6, distance=0, 
               pot_height_har=25, dx=0):
    print("\n(1) Solving time-independent Schrodinger equation...")
    ham_op = hamiltonian(ngridx, laplacianDim, pot_height_har, pot_shape, 
                         curvature, width, slope, distance, dx,)
    (eigval, eigvec) = np.linalg.eigh(ham_op)
    print("\nDone!!")
    return eigval, eigvec

def compute_energy(eigval, eigvec, num_state, pot_shape, curvature, width, slope, 
                   distance, pot_height_har, dx):
    '''
    T_op = - (1 / 2) laplacian,  V_op = diag(potential)
    '''
    T_op = -laplacian(ngridx, laplacianDim, dx) / 2.0
    V_1d = potential(ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx)
    V_op = np.diag(V_1d)

    pot = V_1d * eff_har2ev

    if pot_shape == 4: # Exception : triangular
        triangular_max_eV = np.max(pot[1:-1])
        filtered_indices = [i for i in range(len(eigval)) if eigval[i] * eff_har2ev < 1.1 * triangular_max_eV]
    elif pot_shape != 1:  
        filtered_indices = [i for i in range(len(eigval)) if eigval[i] * eff_har2ev < 1.5 * pot_height_har * eff_har2ev]
    else:# Exception : infinite potential
        filtered_indices = [i for i in range(len(eigval)) if eigval[i] * eff_har2ev < 1e2 * pot_height_har * eff_har2ev]
    

    num_to_print = min(num_state, len(filtered_indices))
    print("\nEnergy components for eigenstates :")
    for idx in range(num_to_print):
        j = filtered_indices[idx]
        psi = eigvec[:, j]
        T_exp = np.vdot(psi, T_op @ psi).real
        V_exp = np.vdot(psi, V_op @ psi).real
        total = T_exp + V_exp
        print(f"State {j+1}: Kinetic = {T_exp * eff_har2ev:.2f} eV, Potential = {V_exp * eff_har2ev:.2f} eV, Total = {total * eff_har2ev:.3f} eV")



# (2) Visualization & interactive widget

In [ ]:

# Animation Function: Animate the time evolution of multiple eigenstates' real, imaginary, and probability density
# Only animate eigenstates with eigenenergy below the potential height.
def animate(eigval, eigvec, num_state, boxL, ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx, x_max, y_max, sim_time):
    print("\n(2) Visualizing ...")
    print("\tRed \t= Real part of Psi \n\tBlue \t= Imaginary part of Psi \n\tGrey shaded = Probability Psi*Psi")

    # Define the x-axis in Angstrom units
    x = np.linspace(-boxL/2, boxL/2, ngridx) / eff_ang2bohr
    pot = potential(ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx) * eff_har2ev
    scale = 0.8 * pot_height_har * eff_har2ev  # eV, coefficient for visualization

    # Filter eigenstates with eigenenergy below the potential height
    if pot_shape == 4:  # Exception : triangular
        triangular_max_eV = np.max(pot[1:-1])
        filtered_indices = [i for i in range(len(eigval)) if eigval[i] * eff_har2ev < 1.1 * triangular_max_eV]
    elif pot_shape != 1:
        filtered_indices = [i for i in range(len(eigval)) if eigval[i] * eff_har2ev < 1.5 * pot_height_har * eff_har2ev]
    else:  # Exception : infinite potential
        filtered_indices = [i for i in range(len(eigval)) if eigval[i] * eff_har2ev < 1e2 * pot_height_har * eff_har2ev]
    num_to_plot = min(num_state, len(filtered_indices))

    # Compute normalization and energy offsets for each filtered eigenstate
    len_psi_list = [np.sum(np.abs(eigvec[:, j])**2) for j in filtered_indices[:num_to_plot]]
    E_eV_list = [eigval[j] * eff_har2ev for j in filtered_indices[:num_to_plot]]
    E_J_list = [eigval[j] * physical_constants["Hartree energy"][0] for j in filtered_indices[:num_to_plot]]

    sim_time_s = sim_time * 1e-15  # Convert fsec to seconds
    time_step = 2e-17              # 0.02 fsec
    t_values = np.arange(0, sim_time_s + 1e-20, time_step)

    def _lightness_by_rank(idx, n, low=35.0, high=70.0):
        if n <= 1:
            return 0.5 * (low + high)
        return low + (high - low) * (idx / (n - 1))

    real_colors = [
        f"hsl(6, 90%, {_lightness_by_rank(i, num_to_plot):.1f}%)"
        for i in range(num_to_plot)
    ]
    imag_colors = [
        f"hsl(215, 88%, {_lightness_by_rank(i, num_to_plot):.1f}%)"
        for i in range(num_to_plot)
    ]
    fill_colors = [
        f"rgba(120,120,120,{(0.24 + 0.18 * (0.0 if num_to_plot <= 1 else i / (num_to_plot - 1))):.3f})"
        for i in range(num_to_plot)
    ]

    def _state_curves(state_idx, t):
        j = filtered_indices[state_idx]
        phase = np.exp(-1j * E_J_list[state_idx] * t / hbar)
        psi_t = eigvec[:, j] * phase

        real_scaled = psi_t.real / len_psi_list[state_idx] * scale + E_eV_list[state_idx]
        imag_scaled = psi_t.imag / len_psi_list[state_idx] * scale + E_eV_list[state_idx]
        prob_scaled = ((psi_t.real)**2 + (psi_t.imag)**2) / len_psi_list[state_idx] * (scale * 6) + E_eV_list[state_idx]

        baseline = np.full_like(x, E_eV_list[state_idx])
        x_fill = np.concatenate([x, x[::-1]])
        y_fill = np.concatenate([prob_scaled, baseline[::-1]])
        return real_scaled, imag_scaled, x_fill, y_fill

    static_annotations = []
    x_text = 8  # in Angstrom units
    for idx in range(num_to_plot):
        if (E_eV_list[idx] + y_max / 40.0) < y_max:
            static_annotations.append(
                dict(
                    x=x_text,
                    y=E_eV_list[idx] + y_max / 40.0,
                    text=f"{E_eV_list[idx]:.2f} eV",
                    showarrow=False,
                    font=dict(size=10, color="black"),
                    xanchor="left",
                    yanchor="middle",
                )
            )

    initial_data = [
        go.Scatter(
            x=x,
            y=pot,
            mode="lines",
            name="Potential",
            line=dict(color="black", width=1.2),
        )
    ]

    for idx in range(num_to_plot):
        real_scaled, imag_scaled, x_fill, y_fill = _state_curves(idx, t_values[0])
        initial_data.append(
            go.Scatter(
                x=x,
                y=real_scaled,
                mode="lines",
                name=f"Re(Psi) state {idx + 1}",
                line=dict(color=real_colors[idx], width=1.5),
            )
        )
        initial_data.append(
            go.Scatter(
                x=x,
                y=imag_scaled,
                mode="lines",
                name=f"Im(Psi) state {idx + 1}",
                line=dict(color=imag_colors[idx], width=1.5),
            )
        )
        initial_data.append(
            go.Scatter(
                x=x_fill,
                y=y_fill,
                mode="lines",
                name=f"|Psi|^2 state {idx + 1}",
                line=dict(color="rgba(0,0,0,0)", width=0),
                fill="toself",
                fillcolor=fill_colors[idx],
            )
        )

    frames = []
    dynamic_trace_indices = list(range(1, 1 + 3 * num_to_plot))
    for iframe, t in enumerate(t_values):
        frame_data = []
        for idx in range(num_to_plot):
            real_scaled, imag_scaled, x_fill, y_fill = _state_curves(idx, t)
            frame_data.append(
                go.Scatter(
                    x=x,
                    y=real_scaled,
                    mode="lines",
                    line=dict(color=real_colors[idx], width=1.5),
                )
            )
            frame_data.append(
                go.Scatter(
                    x=x,
                    y=imag_scaled,
                    mode="lines",
                    line=dict(color=imag_colors[idx], width=1.5),
                )
            )
            frame_data.append(
                go.Scatter(
                    x=x_fill,
                    y=y_fill,
                    mode="lines",
                    line=dict(color="rgba(0,0,0,0)", width=0),
                    fill="toself",
                    fillcolor=fill_colors[idx],
                )
            )

        frames.append(
            go.Frame(
                data=frame_data,
                traces=dynamic_trace_indices,
                name=str(iframe),
                layout=go.Layout(title_text=f"Time Evolution of Eigenstates (Time: {t * 1e15:.2f} fsec)"),
            )
        )

    slider_steps = []
    for iframe, t in enumerate(t_values):
        slider_steps.append(
            {
                "args": [[str(iframe)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
                "label": f"{t * 1e15:.2f}",
                "method": "animate",
            }
        )

    if (pot_shape != 5) and (pot_shape != 6):
        x_range = [-x_max, x_max]
    else:
        x_range = [-boxL_ang / 2, -boxL_ang / 2 + x_max]

    fig = go.Figure(data=initial_data, frames=frames)
    fig.update_layout(
        width=920,
        height=700,
        template="plotly_white",
        title=f"Time Evolution of Eigenstates (Time: {t_values[0] * 1e15:.2f} fsec)",
        margin=dict(l=80, r=220, t=90, b=140),
        xaxis=dict(title="Box [Angstrom]", range=x_range),
        yaxis=dict(title="Energy [eV]", range=[-0.125 * y_max, y_max]),
        annotations=static_annotations,
        updatemenus=[
            {
                "type": "buttons",
                "direction": "left",
                "showactive": True,
                "x": 0.0,
                "y": -0.06,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [None, {"fromcurrent": True, "frame": {"duration": 50, "redraw": True}, "transition": {"duration": 0}}],
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}],
                    },
                ],
            }
        ],
        sliders=[
            {
                "active": 0,
                "x": 0.22,
                "y": -0.06,
                "len": 0.76,
                "currentvalue": {"prefix": "Time [fsec]: "},
                "pad": {"t": 40},
                "steps": slider_steps,
            }
        ],
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            bgcolor="rgba(255,255,255,0.85)",
        ),
    )

    display(HTML(fig.to_html(full_html=False, include_plotlyjs='inline', auto_play=False)))

    print("\n")
    print("Done!!")



if __name__=="__main__":
    ### Calculation Setting
    # Grid sampling 
    boxL_ang = 500        # Box Length in Angstrom
    ngridx = 2501         # Number of grid points
    laplacianDim = 3

    dx_ang = boxL_ang / ngridx
    print('(0) Grid setting')
    print(f'box length \t = {boxL_ang:5} Angstrom')
    print(f'number of x grid = {ngridx:5}')
    print(f'dx \t\t = {dx_ang / ang2bohr:5.3f} Angstrom')


    ### Widget
    # Widget setting
    widget_width    = 400
    widget_margin   = 5
    widget_text     = 200
    
    # Widget(1): Calculation
    shape_widget = widgets.Dropdown(
        options=[
            ('Finite square well',          0),
            ('Infinite square well',        1),
            ('Double square well',          2),
            ('Harmonic oscillator',         3),
            ('Triangular well',             4),
            ('Asymmetric square well (1)',  5),
            ('Asymmetric square well (2)',  6),
            ('Custum potential',            7),
        ],
        value=0, description='Potential Shape', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    m_eff_widget = widgets.Dropdown(
        options=[
            ('GaAs : 0.07',  0.07),
            ('Silicon : 1.08', 1.08),
            ('Germanium : 0.55', 0.55),
            ('Bare mass : 1.00',  1.00),
        ],
        value=1.00, description='Effective mass', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    width_widget = widgets.FloatSlider(
        min=1, max=100, step=0.1,value=100, description='Width [Ang]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    height_widget = widgets.FloatSlider(
        value=4, min=0.2, max=30, step=0.1,  description='Height [eV]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    curvature_widget = widgets.FloatSlider(
        min=0.1, max=10, step=0.1, value=1, description='Curvature of harmonic well',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    slope_widget = widgets.FloatSlider(
        min=0.1, max=50, step=0.1, value=3, description='Slope of triangular well [eV/Ang]',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    distance_widget = widgets.FloatSlider(
        min=0, max=100, step=0.2,value=5, description='Distance [Ang]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )

    # Widget(2): Visualization
    num_state_widget = widgets.IntSlider(
        min=1, max=10, step=1, value=5, description='# of States',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{5*widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    x_max_widget = widgets.IntSlider(
        min=1, max=boxL_ang, step=1, value=250, description='x range of plot [Ang]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    y_max_widget = widgets.FloatSlider(
        min=0.2, max=35, step=0.2, value=5, description='y range of plot [eV]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    sim_time_widget = widgets.IntSlider(
        min=0, max=10, step=1, value=0, description='Simulation Time [fsec]',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    
    # Real time visibility control
    def update_visibility(change):
        value = shape_widget.value
        width_widget.layout.display     = 'flex'   if value <= 2 or value >= 4 else 'none'
        distance_widget.layout.display  = 'flex'   if value == 2 or value == 6 else 'none'
        height_widget.layout.display    = 'flex'   if value == 0 or value == 2 or value >= 5 else 'none'
        curvature_widget.layout.display = 'flex'   if value == 3 else 'none'
        slope_widget.layout.display     = 'flex'   if value == 4 else 'none'

    curvature_widget.layout.display = 'none'
    distance_widget.layout.display  = 'none'
    # width_widget.layout.display   = 'none'
    # height_widget.layout.display  = 'none'
    slope_widget.layout.display     = 'none'
    shape_widget.observe(update_visibility, names='value')

    @interact_manual(
        pot_shape = shape_widget,
        m_eff     = m_eff_widget,
        curvature = curvature_widget,
        width     = width_widget,
        slope     = slope_widget,
        pot_height_eV = height_widget,
        distance  = distance_widget,
        

        num_state = num_state_widget,
        x_max     = x_max_widget,
        y_max     = y_max_widget,
        sim_time  = sim_time_widget
    )
    def main(pot_shape, m_eff, curvature, width, slope, pot_height_eV, distance, 
             num_state, x_max, y_max, sim_time):
        
        ### Unit conversion
        '''
        Effective mass atomic unit

        - ref
            Quantum Mechanics with Applications to Nanotechnology and Information Science, Yehuda B. Band, Yshai Avishai, Ch3 (2013)
            https://www.sciencedirect.com/topics/mathematics/atomic-unit

 
        - Units
            Effective (mass) Bohr radius
                a*   = a0 / m*        (a0 is Bohr radius)

            Effective (mass) Hartree
                E_h* = m* x E_h    (E_h is Hartree energy)

        - Schrodinger equation

            iℏ * ∂​Ψ/∂t(r,t) = ( -( ℏ^2 / 2m* ) ​​∇^2 + V(r,t) ) Ψ(r,t)

                ↓ 

            i * ∂​Ψ/∂t(r,t) = ( -( 1 / 2 ) ​​∇^2 + V(r,t) ) Ψ(r,t)

        '''
        
        global eff_bohr
        global eff_ang2bohr
        global eff_har
        global eff_har2ev

        eff_bohr = bohr_radius / m_eff
        eff_ang2bohr = 1e-10 / eff_bohr
        eff_har = hartree_energy * m_eff
        eff_har2ev = eff_har / ev_energy
        
        dx              = dx_ang    * eff_ang2bohr
        boxL            = boxL_ang  * eff_ang2bohr
        pot_height_har  = pot_height_eV / eff_har2ev
        width_bohr      = width     * eff_ang2bohr
        distance_bohr   = distance  * eff_ang2bohr
        slope_har       = slope     / (eff_har2ev * eff_ang2bohr)    # [Hartree/Bohr] from [eV/Ang]

        
        ### Calculation & Visualization
        (eigval, eigvec) = solve_tise(pot_shape, curvature, width_bohr, slope_har, distance_bohr, pot_height_har, dx)

        compute_energy(eigval, eigvec, num_state, 
                       pot_shape, curvature, width_bohr, slope_har, distance_bohr, pot_height_har, dx)
        
        animate(eigval, eigvec, num_state, 
                boxL, ngridx, pot_shape, pot_height_har, curvature, width_bohr, slope_har, distance_bohr, dx, 
                x_max, y_max, sim_time)

    readme = r"""
              width
       ↑      <---->
height ┼      ┌-----┐
       │      │     │
       │      │     │
       │------┘     └------
       --------------------->

    """
    print(readme)
    pass


(0) Grid setting
box length 	 =   500 Angstrom
number of x grid =  2501
dx 		 = 0.106 Angstrom


interactive(children=(Dropdown(description='Potential Shape', layout=Layout(margin='5px 5px 5px 5px', width='4…


              width
       ↑      <---->
height ┼      ┌-----┐
       │      │     │
       │      │     │
       │------┘     └------
       --------------------->

    
